In [ ]:
import anndata

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation

import matplotlib.pyplot as plt

import os

sc.settings.verbosity = 3            
sc.logging.print_header()
sc.settings.set_figure_params(dpi=1000, facecolor='white', frameon=False, format='pdf')
sc.settings.figdir = "./5_results/"

In [ ]:
#F5f
adata = sc.read_h5ad('../yourpath/4_Tcells.h5ad')
adata = adata[adata.obs['anno'].isin(['CD8 Tn-like', 'CD8 Teff', 'CD8 Tex prog','CD8 Tex int','CD8 Tex term',
                 'CD4 Tn-like','CD4 Teff Gzmk+','CD4 Treg'])]
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10) 
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=25)
sc.tl.umap(adata)
desired_order = ['CD8 Tn-like', 'CD8 Teff', 'CD8 Tex prog','CD8 Tex int','CD8 Tex term',
                 'CD4 Tn-like','CD4 Teff Gzmk+','CD4 Treg']
adata.obs['anno'] = adata.obs['anno'].astype('category')
adata.obs['anno'] = adata.obs['anno'].cat.reorder_categories(desired_order)
adata.uns['anno_colors'] = ['#969696','#54A05D','#A7C8DF','#DAD9E9','#9E9AC4',
                            '#F2B275','#D55E2A','#4880B8']
sc.pl.umap(adata, color=['anno'], vmax='p99', save='../yourpath/Tcell.umap.pdf')

In [ ]:
#S5g
sc.settings.set_figure_params(dpi=1000, facecolor='white', frameon=False, format='pdf', figsize=(8, 4))

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
adata_sub2 = sc.read_h5ad('../yourpath/4_Tcells.h5ad')
adata_sub2 = adata_sub2[adata_sub2.obs['anno'].isin(['CD8 Tn-like','CD8 Teff', 'CD8 Tex prog','CD8 Tex int','CD8 Tex term'])]
adata_sub2.layers["counts"] = adata_sub2.X.copy()
sc.pp.normalize_total(adata_sub2, target_sum=1e4)
sc.pp.log1p(adata_sub2)
adata_sub2.raw = adata_sub2
adata_sub2.layers["scaled"] = sc.pp.scale(adata_sub2, copy=True).X
desired_order = ['CD8 Tn-like','CD8 Teff', 'CD8 Tex prog','CD8 Tex int','CD8 Tex term']
adata_sub2.obs['anno'] = adata_sub2.obs['anno'].astype('category')
adata_sub2.obs['anno'] = adata_sub2.obs['anno'].cat.reorder_categories(desired_order)

colors = ["#00007A", "white", "#79150D"]
values = [-100, 0, 100]
norm_values = [(v - min(values)) / (max(values) - min(values)) for v in values]
custom_cmap = LinearSegmentedColormap.from_list("custom_cmap", list(zip(norm_values, colors)))
sc.pl.matrixplot(
    adata_sub2,
    [
        'Sell', 'Il7r', 'Tcf7', 'Slamf6',
        'Ccl5', 'Gzmk',
        'Eomes',
        'Mki67', 'Top2a',
        'Cx3cr1', 'Xcl1',
        'Ctla4', 'Tigit', 'Tox', 'Lag3', 'Pdcd1', 'Havcr2'
    ],
    groupby="anno",
    colorbar_title="Z Score",
    layer="scaled",
    vmin=-0.5,
    vmax=0.5,
    cmap=custom_cmap,
    save="../yourpath/Tcell.matrixplot.custom_cmap.pdf"
)